## Part 1 - Fetch And Pre-Filter OSM Railways

This cell queries Overpass and keeps conventional rail ways in the corridor bbox, excluding obvious high-speed (AVE) and yard/service tracks.

In [4]:
overpass_query = f"""
[out:json][timeout:180];
(
  way[\"railway\"=\"rail\"]({S},{W},{N},{E});
);
(._;>;);
out body;
"""

data, endpoint_used = fetch_overpass_json(overpass_query, OVERPASS_ENDPOINTS)
print(f"Overpass endpoint used: {endpoint_used}")

nodes = {}
raw_ways = []
for el in data["elements"]:
    if el["type"] == "node":
        nodes[el["id"]] = {"lat": el["lat"], "lon": el["lon"]}
    elif el["type"] == "way":
        raw_ways.append(el)

ways = []
excluded_highspeed = 0
for w in raw_ways:
    tags = w.get("tags", {})
    if tags.get("railway") != "rail":
        continue
    if tags.get("service") in EXCLUDED_SERVICE:
        continue
    if tags.get("usage") == "industrial":
        continue
    if is_high_speed_way(tags):
        excluded_highspeed += 1
        continue
    ways.append(w)

print(f"Loaded nodes: {len(nodes)}")
print(f"Candidate rail ways (non-AVE): {len(ways)}")
print(f"Excluded high-speed ways: {excluded_highspeed}")

Endpoint failed: https://overpass-api.de/api/interpreter -> 406 Client Error: Not Acceptable for url: https://overpass-api.de/api/interpreter?data=%0A%5Bout%3Ajson%5D%5Btimeout%3A180%5D%3B%0A%28%0A++way%5B%22railway%22%3D%22rail%22%5D%2841.2%2C2.0%2C42.55%2C3.35%29%3B%0A%29%3B%0A%28._%3B%3E%3B%29%3B%0Aout+body%3B%0A
Overpass endpoint used: https://overpass.kumi.systems/api/interpreter
Loaded nodes: 26100
Candidate rail ways (non-AVE): 2388
Excluded high-speed ways: 600


## Part 2 - Select All Relevant Sections In The Barcelona-Cerbere Network

This cell builds the rail graph, keeps the connected component that links Barcelona Sants to Cerbere, and then keeps all ways that are within a corridor around the Barcelona-Cerbere axis.

In [5]:
graph = build_graph(ways, nodes)
node_ids = list(graph.keys())

start = nearest_node(*BARCELONA_SANTS, node_ids=node_ids, nodes=nodes)
goal = nearest_node(*CERBERE, node_ids=node_ids, nodes=nodes)

component_nodes = connected_component(start, graph)
if goal not in component_nodes:
    raise RuntimeError(
        "Barcelona Sants and Cerbere are not connected in the filtered conventional rail network."
    )

component_ways = []
for w in ways:
    if any(nid in component_nodes for nid in w.get("nodes", [])):
        component_ways.append(w)

selected_ways = []
for w in component_ways:
    if way_in_corridor(
        w,
        nodes,
        BARCELONA_SANTS[0],
        BARCELONA_SANTS[1],
        CERBERE[0],
        CERBERE[1],
        CORRIDOR_HALF_WIDTH_KM,
    ):
        selected_ways.append(w)

section_rows = []
for w in selected_ways:
    tags = w.get("tags", {})
    section_rows.append(
        {
            "way_id": w["id"],
            "name": tags.get("name"),
            "operator": tags.get("operator"),
            "gauge": tags.get("gauge"),
            "electrified": tags.get("electrified"),
            "maxspeed": tags.get("maxspeed"),
            "usage": tags.get("usage"),
            "service": tags.get("service"),
            "highspeed": tags.get("highspeed"),
            "line": tags.get("line"),
            "ref": tags.get("ref"),
        }
    )

sections_df = pd.DataFrame(section_rows).drop_duplicates(subset=["way_id"])

print(f"Component nodes: {len(component_nodes)}")
print(f"Component ways: {len(component_ways)}")
print(f"Selected corridor ways: {len(selected_ways)}")
sections_df.head(20)

Component nodes: 7447
Component ways: 1431
Selected corridor ways: 1397


,way_id,name,operator,gauge,electrified,maxspeed,usage,service,highspeed,line,ref
0,5172926,FFCC Cerbère - Barcelona,Adif,1668,contact_line,140,main,None,None,None,270
1,6245707,FFCC Barcelona-Maçanet via Mataró,Adif,1668,contact_line,140,main,None,None,None,R1;RG1
2,6245871,FFCC Barcelona-Maçanet via Mataró,Adif,1668,contact_line,140,main,None,None,None,R1;RG1
3,6246102,FFCC Barcelona-Maçanet via Mataró,Adif,1668,contact_line,140,main,None,None,None,R1;RG1
4,6246108,FFCC Barcelona-Maçanet via Mataró,Adif,1668,contact_line,140,main,None,None,None,R1;RG1
5,8873194,FFCC Barcelona-Maçanet via Mataró,Adif,1668,contact_line,75,main,None,None,None,R1;RG1
6,18657125,FFCC Barcelona-Vilafranca-Sant Vicenç de Calders,Adif,1668,contact_line,70,main,None,None,None,R1;R3;R4;RG1
7,19918614,Línia FFCC Ramal Aeroport R2Nord,Adif,1668,contact_line,90,branch,None,None,None,254
8,19979784,FFCC Barcelona - Tarragona via Vilanova i la G...,Adif,1668,contact_line,160,main,None,None,None,R2;R2Sud;R13;R14;R15;R16
9,20169326,FFCC Barcelona - Tarragona via Vilanova i la G...,Adif,1668,contact_line,160,main,None,None,None,R2;R2Sud;R13;R14;R15;R16


## Part 3 - Export Corridor Sections To GeoJSON And CSV

This cell writes all selected railway sections (not just a single route line) to files.

In [6]:
out_dir = Path(".")
out_geojson = out_dir / "barcelona_cerbere_corridor_sections.geojson"
out_csv = out_dir / "barcelona_cerbere_corridor_sections.csv"

features = []
for w in selected_ways:
    wn = [nid for nid in w.get("nodes", []) if nid in nodes]
    if len(wn) < 2:
        continue
    coords = [[nodes[nid]["lon"], nodes[nid]["lat"]] for nid in wn]
    tags = w.get("tags", {})
    features.append(
        {
            "type": "Feature",
            "properties": {
                "from": "Barcelona Sants",
                "to": "Cerbere",
                "way_id": w["id"],
                "name": tags.get("name"),
                "operator": tags.get("operator"),
                "gauge": tags.get("gauge"),
                "electrified": tags.get("electrified"),
                "maxspeed": tags.get("maxspeed"),
                "usage": tags.get("usage"),
                "service": tags.get("service"),
                "highspeed": tags.get("highspeed"),
                "line": tags.get("line"),
                "ref": tags.get("ref"),
            },
            "geometry": {"type": "LineString", "coordinates": coords},
        }
    )

feature_collection = {"type": "FeatureCollection", "features": features}

out_geojson.write_text(json.dumps(feature_collection, ensure_ascii=True, indent=2))
sections_df.to_csv(out_csv, index=False)

print(f"Section features: {len(features)}")
print(f"GeoJSON: {out_geojson.resolve()}")
print(f"CSV: {out_csv.resolve()}")

Section features: 1397
GeoJSON: /Users/ivo/Documents/Projects/RailSignal/data/raw/barcelona_cerbere_corridor_sections.geojson
CSV: /Users/ivo/Documents/Projects/RailSignal/data/raw/barcelona_cerbere_corridor_sections.csv


## Part 4 - Interactive Map With GeoPandas

This cell uses GeoPandas `explore()` for an interactive Leaflet map.

In [7]:
from pathlib import Path
import importlib
import subprocess
import sys

for pkg in ["geopandas", "folium", "mapclassify"]:
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

import geopandas as gpd

geojson_path = Path("barcelona_cerbere_corridor_sections.geojson")
if not geojson_path.exists():
    raise FileNotFoundError(
        "Run cells 1 through 7 first to create barcelona_cerbere_corridor_sections.geojson."
    )

gdf = gpd.read_file(geojson_path)
if gdf.crs is None:
    gdf = gdf.set_crs(epsg=4326)

gdf.explore(
    tooltip=["way_id", "name", "operator", "maxspeed", "usage", "highspeed"],
    tiles="OpenStreetMap",
    style_kwds={"color": "#c1121f", "weight": 3, "opacity": 0.85},
)

## Part 5 - GTFS Exploration For RENFE R11

This section uses gtfs-kit to inspect R11 route variants (regional and media distancia) and derive stop-to-stop sections from GTFS.

In [13]:
from pathlib import Path
import importlib
import subprocess
import sys

for pkg in ["gtfs_kit"]:
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "gtfs-kit"])

import gtfs_kit as gk
import pandas as pd

gtfs_zip = Path("..") / "f-cercanías~renfe-latest.zip"
if not gtfs_zip.exists():
    raise FileNotFoundError(f"GTFS zip not found: {gtfs_zip.resolve()}")

feed = gk.read_feed(str(gtfs_zip), dist_units="km")

routes = feed.routes.copy()
for c in ["route_id", "route_short_name", "route_long_name", "route_desc"]:
    if c not in routes.columns:
        routes[c] = ""
routes["route_id"] = routes["route_id"].fillna("").astype(str).str.strip()
routes["route_short_name"] = routes["route_short_name"].fillna("").astype(str).str.strip()

r11_routes = routes[routes["route_short_name"].str.upper() == "R11"].copy()
if r11_routes.empty:
    raise RuntimeError("No routes with route_short_name = R11 found in this feed.")

print(f"Total routes in feed: {len(routes)}")
print(f"R11 route rows found: {len(r11_routes)}")

trips = feed.trips.copy()
for c in ["trip_id", "route_id", "trip_headsign", "direction_id", "service_id"]:
    if c not in trips.columns:
        trips[c] = ""
trips["trip_id"] = trips["trip_id"].fillna("").astype(str).str.strip()
trips["route_id"] = trips["route_id"].fillna("").astype(str).str.strip()

shared_route_ids = set(r11_routes["route_id"]).intersection(set(trips["route_id"]))
print(f"R11 route_ids present in trips: {len(shared_route_ids)}")

trips_r11 = trips.merge(
    r11_routes[["route_id", "route_short_name", "route_long_name", "route_desc"]],
    on="route_id",
    how="inner",
)

for c in ["route_long_name", "route_desc", "trip_headsign", "direction_id"]:
    if c not in trips_r11.columns:
        trips_r11[c] = ""

trips_r11["service_family"] = "other"
mask_regional = (
    trips_r11["route_long_name"].fillna("").str.contains("regional", case=False)
    | trips_r11["route_desc"].fillna("").str.contains("regional", case=False)
    | trips_r11["trip_headsign"].fillna("").str.contains("regional", case=False)
)
mask_md = (
    trips_r11["route_long_name"].fillna("").str.contains("media distancia|md", case=False, regex=True)
    | trips_r11["route_desc"].fillna("").str.contains("media distancia|md", case=False, regex=True)
    | trips_r11["trip_headsign"].fillna("").str.contains("media distancia|md", case=False, regex=True)
)
trips_r11.loc[mask_regional, "service_family"] = "regional"
trips_r11.loc[mask_md, "service_family"] = "media_distancia"

variant_cols = [
    "route_id",
    "route_short_name",
    "route_long_name",
    "route_desc",
    "direction_id",
    "service_family",
]
variant_summary = (
    trips_r11.groupby(variant_cols, dropna=False)
    .agg(num_trips=("trip_id", "nunique"), num_services=("service_id", "nunique"))
    .reset_index()
    .sort_values(["service_family", "num_trips"], ascending=[True, False])
)

variant_summary.to_csv("r11_variant_summary.csv", index=False)
print("Saved: r11_variant_summary.csv")
print(f"R11 trips found: {trips_r11['trip_id'].nunique()}")
variant_summary.head(30)

Total routes in feed: 604
R11 route rows found: 4
R11 route_ids present in trips: 2
Saved: r11_variant_summary.csv
R11 trips found: 1369


,route_id,route_short_name,route_long_name,route_desc,direction_id,service_family,num_trips,num_services
0,51T0093R11,R11,Figueres -Barce...,,,other,727,30
1,51T0094R11,R11,Barcelona-Sants -Figue...,,,other,642,30


In [22]:
# Build stop-to-stop sections for R11 and aggregate by service family.
stop_times = feed.stop_times.copy()
stops = feed.stops.copy()

for c in ["trip_id", "stop_id", "stop_sequence"]:
    if c not in stop_times.columns:
        raise RuntimeError(f"Missing required stop_times column: {c}")

stop_times["trip_id"] = stop_times["trip_id"].fillna("").astype(str).str.strip()
stop_times["stop_id"] = stop_times["stop_id"].fillna("").astype(str).str.strip()
stop_times["stop_sequence"] = pd.to_numeric(stop_times["stop_sequence"], errors="coerce")
stop_times = stop_times.dropna(subset=["stop_sequence"])

trip_cols = [
    "trip_id",
    "route_id",
    "direction_id",
    "service_family",
    "route_long_name",
    "route_desc",
    "trip_headsign",
]
trips_r11_local = trips_r11.copy()
trips_r11_local["trip_id"] = trips_r11_local["trip_id"].fillna("").astype(str).str.strip()

edges = stop_times.merge(trips_r11_local[trip_cols], on="trip_id", how="inner")
print(f"R11 stop_times rows after join: {len(edges)}")

edges = edges.sort_values(["trip_id", "stop_sequence"])
edges["to_stop_id"] = edges.groupby("trip_id")["stop_id"].shift(-1)
edges["to_sequence"] = edges.groupby("trip_id")["stop_sequence"].shift(-1)
edges = edges.dropna(subset=["to_stop_id"])

stops_local = stops.copy()
for c in ["stop_id", "stop_name"]:
    if c not in stops_local.columns:
        raise RuntimeError(f"Missing required stops column: {c}")

stops_local["stop_id"] = stops_local["stop_id"].fillna("").astype(str).str.strip()
name_map = stops_local[["stop_id", "stop_name"]].drop_duplicates()

edges = edges.merge(
    name_map.rename(columns={"stop_id": "from_stop_id", "stop_name": "from_stop_name"}),
    left_on="stop_id",
    right_on="from_stop_id",
    how="left",
)
edges = edges.merge(
    name_map.rename(columns={"stop_id": "to_stop_id_map", "stop_name": "to_stop_name"}),
    left_on="to_stop_id",
    right_on="to_stop_id_map",
    how="left",
)

sections = (
    edges.groupby(
        [
            "service_family",
            "route_id",
            "direction_id",
            "stop_id",
            "to_stop_id",
            "from_stop_name",
            "to_stop_name",
        ],
        dropna=False,
    )
    .agg(
        num_trips=("trip_id", "nunique"),
        first_stop_sequence=("stop_sequence", "min"),
        last_stop_sequence=("to_sequence", "max"),
    )
    .reset_index()
    .sort_values(
        ["service_family", "route_id", "direction_id", "first_stop_sequence", "num_trips"],
        ascending=[True, True, True, True, False],
    )
)

sections = sections.rename(columns={"stop_id": "from_stop_id"})
sections.to_csv("r11_sections_by_variant.csv", index=False)
print("Saved: r11_sections_by_variant.csv")
print(f"R11 sections rows: {len(sections)}")

sections.head(40)

R11 stop_times rows after join: 22658
Saved: r11_sections_by_variant.csv
R11 sections rows: 77


,service_family,route_id,direction_id,from_stop_id,to_stop_id,from_stop_name,to_stop_name,num_trips,first_stop_sequence,last_stop_sequence
16,other,51T0093R11,,79300,79203,Girona,Caldes De Malavella,400,1,6
26,other,51T0093R11,,79309,79303,Figueres,Flaçà,357,1,4
27,other,51T0093R11,,79309,79308,Figueres,Vilamalla,305,1,6
37,other,51T0093R11,,79315,79314,Portbou,Colera,245,1,2
36,other,51T0093R11,,79315,79312,Portbou,Llançà,106,1,2
28,other,51T0093R11,,79309,79311,Figueres,Vilajuïga,22,1,2
13,other,51T0093R11,,79203,79202,Caldes De Malavella,Sils,705,2,17
20,other,51T0093R11,,79303,79300,Flaçà,Girona,357,2,5
25,other,51T0093R11,,79308,79306,Vilamalla,Sant Miquel De Fluvià,305,2,7
34,other,51T0093R11,,79314,79312,Colera,Llançà,245,2,3


In [16]:
# Derive R11 service patterns to separate regional vs media-distancia behavior.
trip_stop_counts = (
    stop_times[stop_times["trip_id"].isin(trips_r11["trip_id"])].groupby("trip_id")["stop_id"].count().reset_index(name="num_stops")
)

trip_terminals = (
    stop_times[stop_times["trip_id"].isin(trips_r11["trip_id"])].sort_values(["trip_id", "stop_sequence"]) 
)
first_stops = trip_terminals.groupby("trip_id").first().reset_index()[["trip_id", "stop_id"]].rename(columns={"stop_id": "first_stop_id"})
last_stops = trip_terminals.groupby("trip_id").last().reset_index()[["trip_id", "stop_id"]].rename(columns={"stop_id": "last_stop_id"})

trip_profiles = trips_r11[["trip_id", "route_id", "direction_id", "trip_headsign", "service_family"]].copy()
trip_profiles = trip_profiles.merge(trip_stop_counts, on="trip_id", how="left")
trip_profiles = trip_profiles.merge(first_stops, on="trip_id", how="left")
trip_profiles = trip_profiles.merge(last_stops, on="trip_id", how="left")

pattern_summary = (
    trip_profiles.groupby(["route_id", "direction_id", "first_stop_id", "last_stop_id"], dropna=False)
    .agg(
        num_trips=("trip_id", "nunique"),
        median_num_stops=("num_stops", "median"),
        min_num_stops=("num_stops", "min"),
        max_num_stops=("num_stops", "max"),
    )
    .reset_index()
    .sort_values(["route_id", "direction_id", "num_trips"], ascending=[True, True, False])
)

pattern_summary.to_csv("r11_pattern_summary.csv", index=False)
print("Saved: r11_pattern_summary.csv")
print("Use median_num_stops + terminals to tag each pattern as regional or media_distancia.")
pattern_summary.head(30)

Saved: r11_pattern_summary.csv
Use median_num_stops + terminals to tag each pattern as regional or media_distancia.


,route_id,direction_id,first_stop_id,last_stop_id,num_trips,median_num_stops,min_num_stops,max_num_stops
5,51T0093R11,,79315,71801,351,27.0,12,27
1,51T0093R11,,79309,71801,251,10.0,10,10
2,51T0093R11,,79309,79200,50,14.0,14,14
0,51T0093R11,,79300,71801,43,8.0,8,8
3,51T0093R11,,79309,79315,22,5.0,5,5
4,51T0093R11,,79309,79607,10,15.0,15,15
8,51T0094R11,,71801,79315,340,29.0,12,29
7,51T0094R11,,71801,79309,245,10.0,10,10
6,51T0094R11,,71801,79300,50,8.0,8,8
9,51T0094R11,,79315,79309,7,5.0,5,5


## Part 6 - R11 Trajectory-First View

This cell focuses on full route trajectories. It tags the route with fewer median stops as media_distancia, then builds one representative trajectory per R11 route (prefer GTFS shapes, fallback to stop chain).

In [25]:
from pathlib import Path

import pandas as pd

# Ensure required package for geometry serialization is available
import importlib
import subprocess
import sys

for pkg in ["shapely", "geopandas"]:
    try:
        importlib.import_module(pkg)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

from shapely.geometry import LineString
import geopandas as gpd

# 1) Heuristic label: fewer stops => media_distancia (route-level)
route_stop_stats = (
    trip_profiles.groupby("route_id", dropna=False)
    .agg(median_num_stops=("num_stops", "median"), num_trips=("trip_id", "nunique"))
    .reset_index()
)

route_stop_stats = route_stop_stats.sort_values("median_num_stops")
route_stop_stats["service_guess"] = "unknown"
if len(route_stop_stats) >= 2:
    min_stops = route_stop_stats["median_num_stops"].min()
    candidates = route_stop_stats[route_stop_stats["median_num_stops"] == min_stops]
    if len(candidates) == 1:
        md_route_id = candidates.iloc[0]["route_id"]
        route_stop_stats["service_guess"] = route_stop_stats["route_id"].apply(
            lambda r: "media_distancia" if r == md_route_id else "regional"
        )

# 2) Build representative geometry per route_id
trips_r11_geo = trips_r11.copy()
if "shape_id" not in trips_r11_geo.columns:
    trips_r11_geo["shape_id"] = ""
trips_r11_geo["shape_id"] = trips_r11_geo["shape_id"].fillna("").astype(str).str.strip()

shapes_df = getattr(feed, "shapes", None)
if shapes_df is None:
    shapes_df = pd.DataFrame()

geom_rows = []
for route_id in route_stop_stats["route_id"].tolist():
    route_trips = trips_r11_geo[trips_r11_geo["route_id"] == route_id].copy()

    geom = None
    geom_source = None

    # Preferred: most common shape_id for this route
    if not shapes_df.empty and "shape_id" in shapes_df.columns:
        shape_counts = route_trips[route_trips["shape_id"] != ""]["shape_id"].value_counts()
        if len(shape_counts) > 0:
            top_shape_id = shape_counts.index[0]
            shp = shapes_df[shapes_df["shape_id"].astype(str) == str(top_shape_id)].copy()
            if {"shape_pt_lon", "shape_pt_lat"}.issubset(shp.columns):
                if "shape_pt_sequence" in shp.columns:
                    shp = shp.sort_values("shape_pt_sequence")
                coords = list(zip(shp["shape_pt_lon"].astype(float), shp["shape_pt_lat"].astype(float)))
                if len(coords) >= 2:
                    geom = LineString(coords)
                    geom_source = f"shape:{top_shape_id}"

    # Fallback: most common trip stop chain
    if geom is None:
        rt_ids = set(route_trips["trip_id"].astype(str))
        st = stop_times[stop_times["trip_id"].astype(str).isin(rt_ids)].copy()
        if len(st) > 0:
            st["stop_sequence"] = pd.to_numeric(st["stop_sequence"], errors="coerce")
            st = st.dropna(subset=["stop_sequence"]).sort_values(["trip_id", "stop_sequence"])
            chain_counts = (
                st.groupby("trip_id")["stop_id"].apply(lambda s: tuple(s.astype(str))).value_counts()
            )
            if len(chain_counts) > 0:
                top_chain = chain_counts.index[0]
                stop_coords = stops[["stop_id", "stop_lon", "stop_lat"]].copy()
                stop_coords["stop_id"] = stop_coords["stop_id"].astype(str)
                coord_map = {
                    sid: (float(lon), float(lat))
                    for sid, lon, lat in stop_coords[["stop_id", "stop_lon", "stop_lat"]].itertuples(index=False)
                    if pd.notna(lon) and pd.notna(lat)
                }
                coords = [coord_map[sid] for sid in top_chain if sid in coord_map]
                if len(coords) >= 2:
                    geom = LineString(coords)
                    geom_source = "stop_chain"

    if geom is not None:
        row = route_stop_stats[route_stop_stats["route_id"] == route_id].iloc[0].to_dict()
        row["geom_source"] = geom_source
        row["geometry"] = geom
        geom_rows.append(row)

traj_gdf = gpd.GeoDataFrame(geom_rows, geometry="geometry", crs="EPSG:4326")

# 3) Export and preview
out_geojson = Path("r11_route_trajectories.geojson")
out_csv = Path("r11_route_trajectories_summary.csv")

traj_gdf.to_file(out_geojson, driver="GeoJSON")
traj_gdf.drop(columns=["geometry"]).to_csv(out_csv, index=False)

print(f"Trajectory rows: {len(traj_gdf)}")
print(f"GeoJSON: {out_geojson.resolve()}")
print(f"CSV: {out_csv.resolve()}")

traj_gdf[["route_id", "service_guess", "median_num_stops", "num_trips", "geom_source"]]

Trajectory rows: 2
GeoJSON: /Users/ivo/Documents/Projects/RailSignal/data/raw/r11_route_trajectories.geojson
CSV: /Users/ivo/Documents/Projects/RailSignal/data/raw/r11_route_trajectories_summary.csv


,route_id,service_guess,median_num_stops,num_trips,geom_source
0,51T0093R11,unknown,12.0,727,shape:51_R11_INV
1,51T0094R11,unknown,12.0,642,shape:51_R11


## Part 7 - Map All Matched Stations (GeoDataFrame Explore)

This cell maps all stations that appear in the matched R11 sections and overlays them on the route trajectories.

In [21]:
import pandas as pd
import geopandas as gpd

if "sections" not in globals() or sections.empty:
    raise RuntimeError("Run the GTFS section extraction cell first so 'sections' is available.")

if "stops" not in globals() or stops.empty:
    raise RuntimeError("Run the GTFS loading cell first so 'stops' is available.")

# Collect all station IDs that appear in matched section endpoints.
matched_station_ids = pd.unique(
    pd.concat([sections["from_stop_id"].astype(str), sections["to_stop_id"].astype(str)], ignore_index=True)
)

# Compute how often a station appears in matched sections for basic ranking/sizing.
station_counts = pd.concat(
    [
        sections[["from_stop_id", "num_trips"]].rename(columns={"from_stop_id": "stop_id"}),
        sections[["to_stop_id", "num_trips"]].rename(columns={"to_stop_id": "stop_id"}),
    ],
    ignore_index=True,
)
station_counts["stop_id"] = station_counts["stop_id"].astype(str)
station_counts = (
    station_counts.groupby("stop_id", dropna=False)
    .agg(section_appearances=("stop_id", "count"), total_trip_weight=("num_trips", "sum"))
    .reset_index()
)

stops_local = stops.copy()
stops_local["stop_id"] = stops_local["stop_id"].astype(str)
matched_stations = stops_local[stops_local["stop_id"].isin(matched_station_ids)].copy()
matched_stations = matched_stations.merge(station_counts, on="stop_id", how="left")
matched_stations["section_appearances"] = matched_stations["section_appearances"].fillna(0).astype(int)
matched_stations["total_trip_weight"] = matched_stations["total_trip_weight"].fillna(0).astype(int)

matched_gdf = gpd.GeoDataFrame(
    matched_stations,
    geometry=gpd.points_from_xy(matched_stations["stop_lon"], matched_stations["stop_lat"]),
    crs="EPSG:4326",
)

if "traj_gdf" in globals() and len(traj_gdf) > 0:
    base_map = traj_gdf.explore(
        tooltip=["route_id", "service_guess", "geom_source"],
        tiles="OpenStreetMap",
        style_kwds={"color": "#005f73", "weight": 4, "opacity": 0.75},
    )
else:
    base_map = None

matched_gdf.explore(
    m=base_map,
    tooltip=["stop_id", "stop_name", "section_appearances", "total_trip_weight"],
    color="#d62828",
    marker_kwds={"radius": 5},
    name="Matched R11 Stations",
)


## Part 8 - Export App GeoJSON Files (Stations + Sections)

This cell exports matched R11 stations and stop-to-stop sections into the mobile app data folder.

In [26]:
from pathlib import Path

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, Point
from shapely.ops import substring

if "sections" not in globals() or sections.empty:
    raise RuntimeError("Run GTFS section extraction first so 'sections' exists.")

if "stops" not in globals() or stops.empty:
    raise RuntimeError("Run GTFS loading first so 'stops' exists.")

app_data_dir = Path("..") / ".." / "apps" / "mobile" / "data"
app_data_dir.mkdir(parents=True, exist_ok=True)

# Build station points (only stations used by matched sections).
stops_local = stops.copy()
stops_local["stop_id"] = stops_local["stop_id"].astype(str)

station_ids = pd.unique(
    pd.concat([sections["from_stop_id"].astype(str), sections["to_stop_id"].astype(str)], ignore_index=True)
)

station_counts = pd.concat(
    [
        sections[["from_stop_id", "num_trips"]].rename(columns={"from_stop_id": "stop_id"}),
        sections[["to_stop_id", "num_trips"]].rename(columns={"to_stop_id": "stop_id"}),
    ],
    ignore_index=True,
)
station_counts["stop_id"] = station_counts["stop_id"].astype(str)
station_counts = (
    station_counts.groupby("stop_id", dropna=False)
    .agg(section_appearances=("stop_id", "count"), total_trip_weight=("num_trips", "sum"))
    .reset_index()
)

stations_df = stops_local[stops_local["stop_id"].isin(station_ids)].copy()
stations_df = stations_df.merge(station_counts, on="stop_id", how="left")
stations_df["section_appearances"] = stations_df["section_appearances"].fillna(0).astype(int)
stations_df["total_trip_weight"] = stations_df["total_trip_weight"].fillna(0).astype(int)

stations_gdf = gpd.GeoDataFrame(
    stations_df,
    geometry=gpd.points_from_xy(stations_df["stop_lon"], stations_df["stop_lat"]),
    crs="EPSG:4326",
)

# Build section lines using full-resolution route trajectories when available.
stop_coord_map = {
    row.stop_id: (float(row.stop_lon), float(row.stop_lat), row.stop_name)
    for row in stops_local[["stop_id", "stop_lon", "stop_lat", "stop_name"]].itertuples(index=False)
    if pd.notna(row.stop_lon) and pd.notna(row.stop_lat)
}

route_geom_map = {}
if "traj_gdf" in globals() and len(traj_gdf) > 0:
    for row in traj_gdf[["route_id", "geometry"]].itertuples(index=False):
        route_geom_map[str(row.route_id)] = row.geometry

# Use sequence span as primary slicing key to preserve route polyline detail.
route_seq_max = (
    sections.groupby("route_id", dropna=False)["last_stop_sequence"].max().to_dict()
)

section_features = []
fallback_project = 0
fallback_direct = 0
for row in sections.itertuples(index=False):
    from_id = str(row.from_stop_id)
    to_id = str(row.to_stop_id)
    if from_id not in stop_coord_map or to_id not in stop_coord_map:
        continue

    from_lon, from_lat, from_name = stop_coord_map[from_id]
    to_lon, to_lat, to_name = stop_coord_map[to_id]

    # Last fallback is direct stop-to-stop.
    geom = LineString([(from_lon, from_lat), (to_lon, to_lat)])
    geom_mode = "direct"

    route_geom = route_geom_map.get(str(row.route_id))
    if route_geom is not None and not route_geom.is_empty and route_geom.geom_type == "LineString":
        max_seq = route_seq_max.get(row.route_id)

        # Primary: normalized sequence slicing on the full route line.
        if max_seq is not None and pd.notna(max_seq) and float(max_seq) > 1:
            start_norm = max(0.0, min(1.0, (float(row.first_stop_sequence) - 1.0) / (float(max_seq) - 1.0)))
            end_norm = max(0.0, min(1.0, (float(row.last_stop_sequence) - 1.0) / (float(max_seq) - 1.0)))

            try:
                seg = substring(route_geom, min(start_norm, end_norm), max(start_norm, end_norm), normalized=True)
                if seg is not None and not seg.is_empty and seg.geom_type == "LineString" and len(seg.coords) >= 2:
                    geom = seg
                    geom_mode = "sequence"
                else:
                    # Secondary: project stop points onto route line.
                    p_from = Point(from_lon, from_lat)
                    p_to = Point(to_lon, to_lat)
                    d_from = route_geom.project(p_from)
                    d_to = route_geom.project(p_to)
                    seg2 = substring(route_geom, min(d_from, d_to), max(d_from, d_to))
                    if seg2 is not None and not seg2.is_empty and seg2.geom_type == "LineString" and len(seg2.coords) >= 2:
                        geom = seg2
                        geom_mode = "project"
                        fallback_project += 1
                    else:
                        fallback_direct += 1
            except Exception:
                # Secondary: project stop points onto route line.
                p_from = Point(from_lon, from_lat)
                p_to = Point(to_lon, to_lat)
                d_from = route_geom.project(p_from)
                d_to = route_geom.project(p_to)
                try:
                    seg2 = substring(route_geom, min(d_from, d_to), max(d_from, d_to))
                    if seg2 is not None and not seg2.is_empty and seg2.geom_type == "LineString" and len(seg2.coords) >= 2:
                        geom = seg2
                        geom_mode = "project"
                        fallback_project += 1
                    else:
                        fallback_direct += 1
                except Exception:
                    fallback_direct += 1
        else:
            fallback_direct += 1
    else:
        fallback_direct += 1

    section_features.append(
        {
            "service_family": row.service_family,
            "route_id": row.route_id,
            "direction_id": row.direction_id,
            "from_stop_id": from_id,
            "to_stop_id": to_id,
            "from_stop_name": from_name,
            "to_stop_name": to_name,
            "num_trips": int(row.num_trips),
            "first_stop_sequence": float(row.first_stop_sequence),
            "last_stop_sequence": float(row.last_stop_sequence),
            "geometry_mode": geom_mode,
            "geometry": geom,
        }
    )

sections_gdf = gpd.GeoDataFrame(section_features, geometry="geometry", crs="EPSG:4326")

out_stations = app_data_dir / "r11_stations.geojson"
out_sections = app_data_dir / "r11_sections.geojson"

stations_gdf.to_file(out_stations, driver="GeoJSON")
sections_gdf.to_file(out_sections, driver="GeoJSON")

print(f"Stations exported: {len(stations_gdf)} -> {out_stations.resolve()}")
print(f"Sections exported: {len(sections_gdf)} -> {out_sections.resolve()}")
print(f"Projection fallback segments: {fallback_project}")
print(f"Direct fallback segments: {fallback_direct}")

Stations exported: 29 -> /Users/ivo/Documents/Projects/RailSignal/apps/mobile/data/r11_stations.geojson
Sections exported: 77 -> /Users/ivo/Documents/Projects/RailSignal/apps/mobile/data/r11_sections.geojson
Projection fallback segments: 0
Direct fallback segments: 0
